# 第 2 週 實作｜反函數微分、反三角與雙曲函數

已知 $f$ 的導數,能不能不解出 $f^{-1}$ 就寫出它的導數?答案是可以——而且本週所有的反三角、反雙曲公式,全都是同一條規則的特例。


### (選用)讓圖表顯示中文


In [ ]:
import matplotlib
# Colab 想顯示中文: !apt-get -qq install fonts-noto-cjk
# 再設 matplotlib.rcParams['font.sans-serif'] = ['Noto Sans CJK TC']
# 本課圖表標籤一律用英文,不裝字型也不會有豆腐字。
matplotlib.rcParams['axes.unicode_minus'] = False


### 環境設定


In [ ]:
import math
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt


## Lab 1｜反函數的導數:不解 f⁻¹ 也能算

觀念 2 說「不必求出 $f^{-1}$ 的公式」。這格用數值方法驗證這句話是真的。


In [ ]:
from scipy.optimize import brentq

f  = lambda x: x**3 + x          # 嚴格遞增 → 一對一
df = lambda x: 3*x**2 + 1

def f_inv(y, lo=-10, hi=10):
    """數值解 f(x) = y,不用任何解析公式"""
    return brentq(lambda x: f(x) - y, lo, hi)

for b in [2.0, 10.0, -4.0]:
    a = f_inv(b)
    by_formula = 1 / df(a)                       # 反函數公式
    h = 1e-6
    numeric = (f_inv(b + h) - f_inv(b - h)) / (2*h)   # 直接對 f_inv 數值微分
    print(f"b={b:6.1f}  a=f^-1(b)={a: .6f}   1/f'(a)={by_formula:.8f}   "
          f"數值微分={numeric:.8f}   差={abs(by_formula-numeric):.2e}")

# 圖:f 與 f^-1 對稱於 y = x
xs = np.linspace(-2, 2, 300)
ys = f(xs)
plt.plot(xs, ys, label='f(x) = x^3 + x')
plt.plot(ys, xs, label='f inverse')
plt.plot(xs, xs, 'k:', lw=1, label='y = x')
plt.xlim(-4, 4); plt.ylim(-4, 4); plt.gca().set_aspect('equal')
plt.legend(); plt.title('A function and its inverse mirror across y = x')
plt.show()

In [ ]:
# TODO 學生練習:把 f 換成 x**5 + 2*x + 1(仍嚴格遞增)
# 驗證 (f^-1)'(4) = 1/7。提示:先確認 f(1) = 4

## Lab 2｜雙曲函數:為什麼 tanh 適合當 activation

畫出三個雙曲函數,並看 $\tanh$ 的導數 $1-\tanh^{2}$ 如何在兩端趨近 0——這就是深度學習裡「梯度消失」的最小範例。


In [ ]:
xs = np.linspace(-4, 4, 400)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(xs, np.sinh(xs), label='sinh')
ax[0].plot(xs, np.cosh(xs), label='cosh')
ax[0].plot(xs, np.tanh(xs), label='tanh')
ax[0].axhline(1, color='gray', ls=':'); ax[0].axhline(-1, color='gray', ls=':')
ax[0].set_ylim(-4, 4); ax[0].legend(); ax[0].set_title('Hyperbolic functions')

a = np.tanh(xs)
ax[1].plot(xs, 1 - a**2, 'C3')
ax[1].set_title("d/dx tanh = 1 - tanh^2  (gradient dies at the ends)")
ax[1].set_xlabel('x')
plt.tight_layout(); plt.show()

print("cosh^2 - sinh^2 (應恆為 1):")
for x in [0.0, 1.0, 5.0, 10.0]:
    print(f"  x={x:5.1f}   {np.cosh(x)**2 - np.sinh(x)**2:.12f}")

print("\ntanh 的導數在兩端有多小:")
for x in [0.0, 1.0, 3.0, 5.0]:
    print(f"  x={x:5.1f}   1 - tanh^2 = {1 - np.tanh(x)**2:.6e}")

## Lab 3｜懸鏈線 vs 拋物線:伽利略錯在哪

觀念 8 說懸鏈線是 $\cosh$ 不是拋物線。兩者到底差多少?把它量出來。


In [ ]:
a = 1.0
xs = np.linspace(-2, 2, 400)
cat  = a * np.cosh(xs / a)          # 懸鏈線
para = 1 + xs**2 / 2                # 同曲率的拋物線(cosh 的前兩項泰勒)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(xs, cat, label='catenary: cosh(x)')
ax[0].plot(xs, para, '--', label='parabola: 1 + x^2/2')
ax[0].legend(); ax[0].set_title('Nearly identical near 0')

ax[1].semilogy(xs, np.abs(cat - para) + 1e-18)
ax[1].set_title('|difference| grows fast away from 0')
ax[1].set_xlabel('x')
plt.tight_layout(); plt.show()

print(f"{'x':>5}  {'cosh(x)':>12}  {'1+x^2/2':>12}  {'差':>12}")
for x in [0.1, 0.5, 1.0, 2.0, 3.0]:
    c, p = math.cosh(x), 1 + x**2/2
    print(f"{x:5.1f}  {c:12.6f}  {p:12.6f}  {abs(c-p):12.6f}")

# 弧長被積式:根號自動消失
x = sp.Symbol('x', real=True)
y = sp.cosh(x)
integrand = sp.simplify(sp.sqrt(1 + sp.diff(y, x)**2))
print("\nsqrt(1 + y'^2) =", integrand, "  (根號消掉了)")

In [ ]:
# TODO 學生練習:把 a 改成 2 和 0.5,看懸鏈線的形狀怎麼變
# a 是什麼的物理意義?(提示:最低點的高度就是 a)